# HANDS-ON EXERCISE 4

## Energy Consumption Analysis

### Industry: Energy / Utilities

### Introduction

In this exercise, Pandas will be used to inspect, clean, transform, and analyse electricity consumption data collected from customers. The analysis will focus on identifying data-quality issues, creating meaningful consumption categories, and examining consumption patterns across customer groups, cities, and energy sources.

The exercise will demonstrate how data cleaning and basic exploratory analysis can transform raw energy records into useful information for business analysis and further investigation.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Energy_Consumption.csv")

In [3]:
df.head()

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
0,EN00001,Abuja,Residential,111.5,Active,Grid Electricity
1,EN00002,Lagos,Residential,104.0,Active,Grid Electricity
2,EN00003,Lagos,Residential,671.5,Active,Grid Electricity
3,EN00004,Abuja,Residential,172.6,Active,Grid Electricity
4,EN00005,Port Harcourt,Small Business,440.9,Active,Grid + Solar


### Part A - Step 2: Dataset Size

The size of the dataset will be examined to determine the total number of customer records and variables available for analysis.

In [4]:
df.shape

(3012, 6)

### Part A - Step 3: Data Types and Missing Values

The structure of the dataset will now be examined using `info()` to determine the data type of each variable and identify any columns with missing values. This will help establish which variables may require further investigation during the data-cleaning stage.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3012 entries, 0 to 3011
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Customer ID          3012 non-null   object
 1   City                 2970 non-null   object
 2   Customer Type        2994 non-null   object
 3   Monthly Consumption  2894 non-null   object
 4   Account Status       2997 non-null   object
 5   Energy Source        2992 non-null   object
dtypes: object(6)
memory usage: 141.3+ KB


### Part A - Step 4: Summary Statistics

Summary statistics will be examined to understand the numerical characteristics of the dataset. Since the `Monthly Consumption` column is currently stored as an object data type, the initial `describe()` output will also help determine whether numerical values require conversion before meaningful statistical analysis can be performed.

In [8]:
df.describe()

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
count,3012,2970,2994,2894,2997,2992
unique,3000,10,5,2508,4,5
top,EN00100,Lagos,Residential,Pending,Active,Grid Electricity
freq,2,643,1548,21,2430,1661


### Part A - Findings

The dataset contains **3,012 customer records and 6 variables**. All six variables are initially stored as object data types, including `Monthly Consumption`, which is expected to contain numerical electricity consumption values. The dataset contains missing observations in `City`, `Customer Type`, `Monthly Consumption`, `Account Status`, and `Energy Source`, while `Customer ID` has no missing values.

The summary statistics also show that `Customer ID` contains 3,000 unique values among 3,012 records, indicating that some customer IDs occur more than once. In addition, `Monthly Consumption` contains text values, with `Pending` being the most frequently occurring value. These issues require further investigation before the dataset can be used for analysis.

### Part B -Step 1: Investigating Missing Values

The dataset will first be examined for missing observations across all variables. Identifying the extent of missing data before making any changes will help determine which cleaning approach is appropriate for each affected column.

In [9]:
df.isnull().sum()

Customer ID              0
City                    42
Customer Type           18
Monthly Consumption    118
Account Status          15
Energy Source           20
dtype: int64

### Missing Value Findings

The dataset contains missing values in five variables. `City` has **42 missing values**, `Customer Type` has **18**, `Monthly Consumption` has **118**, `Account Status` has **15**, and `Energy Source` has **20**. `Customer ID` has no missing values.

These missing observations will not be removed automatically. Each affected variable will be considered separately to determine the most appropriate treatment based on its role in the analysis and the information available in the dataset.

In [10]:
df[df["City"].isnull()].head(20)

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
138,EN00139,NaN,Residential,248.0,Active,Grid + Solar
185,EN00186,NaN,Industrial,18014.6,Active,Grid + Solar
236,EN00237,NaN,Small Business,528.2,Active,Generator
375,EN00376,NaN,Small Business,931.4,Active,Generator
376,EN00377,NaN,Residential,425.6,Active,Grid Electricity
404,EN00405,NaN,Commercial,3941.3,Inactive,Grid Electricity
429,EN00430,NaN,Residential,NaN,Active,Grid Electricity
512,EN00513,NaN,Residential,113.6,Active,Grid Electricity
694,EN00695,NaN,Residential,404.1,Active,Grid Electricity
698,EN00699,NaN,Residential,303.2,Active,Generator


### Handling Missing City Values

The missing `City` values were investigated before cleaning. The affected records contain other customer information, but the available variables do not provide sufficient evidence to determine the correct city reliably. Therefore, the missing city values will be replaced with **`Unknown`** rather than deleting the records or assigning cities without evidence.

In [11]:
df["City"] = df["City"].fillna("Unknown")

In [12]:
df["City"].isnull().sum()

np.int64(0)

### Part B - Step 5: Investigating Monthly Consumption

The `Monthly Consumption` column is expected to contain numerical electricity consumption values. However, the initial inspection showed that it is stored as an object data type. The column will therefore be investigated for unexpected text values before any conversion or removal of records is performed.

In [14]:
df["Monthly Consumption"].value_counts().head(20)

Monthly Consumption
Pending       21
Unknown       19
Error         16
No Reading    12
5,2O0          9
12O            9
0              7
358.5          4
163.2          3
107.6          3
405.4          3
157.3          3
-45            3
173.8          3
138.7          3
412.8          3
265.3          3
110.9          3
121.2          3
163.6          3
Name: count, dtype: int64

### Identifying Non-Numerical Consumption Values

The `Monthly Consumption` column contains a mixture of numerical values and text entries. Some text entries appear to represent missing or unavailable readings, while others may contain typographical errors that could potentially be corrected. The column will first be tested for values that cannot be interpreted as numerical consumption amounts.

In [15]:
pd.to_numeric(df["Monthly Consumption"], errors="coerce").isnull().sum()

np.int64(206)

### Investigating Non-Numerical Values

The conversion test identified 206 records that cannot currently be interpreted as numerical values. This total includes both genuinely missing values and text entries. The problematic non-numerical entries will therefore be identified separately so that correctable values are distinguished from values that should remain missing.

In [16]:
df.loc[
    pd.to_numeric(df["Monthly Consumption"], errors="coerce").isna(),
    "Monthly Consumption"
].value_counts(dropna=False)

Monthly Consumption
NaN           118
Pending        21
Unknown        19
Error          16
No Reading     12
12O             9
5,2O0           9
##VALUE!        2
Name: count, dtype: int64

### Investigating Correctable Values

Two entries appear to contain typographical errors rather than genuinely missing consumption readings. The value `12O` contains the letter `O` in place of the numerical zero and can reasonably be interpreted as **120**. Similarly, `5,2O0` appears to represent **5,200**, with both a comma and the letter `O` used in place of the numerical zero.

These values will be corrected because their intended numerical meaning is sufficiently clear. This preserves valid records instead of unnecessarily treating them as missing.

In [17]:
df[df["Monthly Consumption"].isin(["12O", "5,2O0"])]

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
165,EN00166,Jos,Residential,12O,Suspended,Grid Electricity
191,EN00192,Enugu,Commercial,"5,2O0",Active,Grid Electricity
193,EN00194,Kaduna,Residential,"5,2O0",Active,Grid + Solar
563,EN00564,Aba,Small Business,"5,2O0",Active,Grid Electricity
975,EN00976,Abuja,Industrial,"5,2O0",Active,Grid Electricity
1066,EN01067,Ibadan,Residential,12O,Active,Grid Electricity
1185,EN01186,Kaduna,Residential,12O,Active,Grid + Generator
1204,EN01205,Ibadan,Residential,12O,Inactive,Grid + Solar
1414,EN01415,Ibadan,Residential,12O,Active,Solar
1747,EN01748,Ibadan,Small Business,"5,2O0",Pending,Grid Electricity


### Correcting Typographical Errors

The investigation confirmed that `12O` and `5,2O0` are consistent typographical representations of numerical consumption values. The letter `O` will be replaced with the numerical zero, and the comma in `5,2O0` will be removed. This correction preserves the affected records and allows the values to be converted to numerical data for analysis.

In [18]:
df["Monthly Consumption"] = df["Monthly Consumption"].replace({
    "12O": "120",
    "5,2O0": "5200"
})

In [19]:
df["Monthly Consumption"].isin(["12O", "5,2O0"]).sum()

np.int64(0)

### Handling Invalid Consumption Entries

After correcting the identifiable typographical errors, the remaining text entries in `Monthly Consumption` represent unavailable or invalid readings, including `Pending`, `Unknown`, `Error`, `No Reading`, and `##VALUE!`. Because these entries do not provide a reliable numerical consumption value, they will be converted to missing values rather than assigned arbitrary numbers. This preserves the records while ensuring that invalid values do not distort numerical analysis.

In [20]:
df["Monthly Consumption"] = df["Monthly Consumption"].replace({
    "Pending": np.nan,
    "Unknown": np.nan,
    "Error": np.nan,
    "No Reading": np.nan,
    "##VALUE!": np.nan
})

In [21]:
df["Monthly Consumption"] = pd.to_numeric(
    df["Monthly Consumption"],
    errors="coerce"
)

In [22]:
df["Monthly Consumption"].dtype

dtype('float64')

### Investigating Suspicious Consumption Values

Although the `Monthly Consumption` column has now been converted to a numerical data type, some values may still require investigation. In particular, zero and negative consumption values may represent unusual or invalid readings. These records will be examined before any decision is made about whether they should be retained or treated as invalid.

In [23]:
df[df["Monthly Consumption"] <= 0]

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
173,EN00174,Port Harcourt,Commercial,0.0,Pending,Grid Electricity
378,EN00379,Kano,Commercial,0.0,Active,Grid Electricity
481,EN00482,Enugu,Residential,0.0,Active,Grid + Solar
1296,EN01297,Port Harcourt,Residential,-45.0,Active,Grid Electricity
1756,EN01757,Lagos,Industrial,0.0,Inactive,Grid + Generator
1845,EN01846,Kano,Residential,0.0,Active,Grid Electricity
1978,EN01979,Port Harcourt,Residential,-45.0,Active,Grid Electricity
2353,EN02354,Kaduna,Industrial,0.0,Inactive,Grid Electricity
2447,EN02448,Abuja,Residential,0.0,Active,Grid + Generator
2812,EN02813,Ibadan,Commercial,-45.0,Active,Grid Electricity


### Handling Negative Consumption Values

The investigation identified three records with a monthly consumption value of **-45**. Negative electricity consumption is not a meaningful consumption value in this dataset and is therefore considered an invalid reading. These values will be replaced with missing values rather than being assigned an arbitrary positive value. The zero values will be retained because zero consumption may represent a legitimate period of no recorded usage and requires further investigation rather than automatic deletion.

In [24]:
df.loc[df["Monthly Consumption"] < 0, "Monthly Consumption"] = np.nan

In [25]:
(df["Monthly Consumption"] < 0).sum()

np.int64(0)

### Verifying Consumption Cleaning

The `Monthly Consumption` column will now be checked after the cleaning operations to determine the final number of missing values and confirm that no unexpected text values remain. This provides a quality check before the column is used for analysis.

In [27]:
df["Monthly Consumption"].isnull().sum()

np.int64(191)

In [28]:
df["Monthly Consumption"].dtype

dtype('float64')

### Part B - Step 14: Investigating Duplicate Customer IDs

The initial summary showed that the dataset contains 3,000 unique Customer IDs across 3,012 records. This indicates that some Customer IDs occur more than once. The duplicate records will be investigated before any records are removed to determine whether they represent unnecessary duplicates or legitimate repeated records.

In [29]:
df["Customer ID"].duplicated().sum()

np.int64(12)

### Inspecting Duplicate Customer Records

The dataset contains 12 repeated Customer ID entries. Because repeated IDs may represent legitimate multiple records rather than duplicate rows, the affected records will be examined before any deletion is considered. This will help determine whether the repeated Customer IDs contain identical information or represent separate customer records.

In [30]:
df[df["Customer ID"].duplicated(keep=False)].sort_values("Customer ID")

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
8,EN00009,Ibadan,Residential,NaN,Active,Grid Electricity
3006,EN00009,Ibadan,Residential,NaN,Active,Grid Electricity
99,EN00100,Benin City,Public Institution,588.0,Active,Grid Electricity
3003,EN00100,Benin City,Public Institution,588.0,Active,Grid Electricity
326,EN00327,Enugu,Commercial,2084.9,Active,Grid Electricity
3005,EN00327,Enugu,Commercial,2084.9,Active,Grid Electricity
3011,EN00394,Port Harcourt,Public Institution,634.9,Active,Grid Electricity
393,EN00394,Port Harcourt,Public Institution,634.9,Active,Grid Electricity
3007,EN00736,Abuja,Residential,247.2,Active,Grid + Solar
735,EN00736,Abuja,Residential,247.2,Active,Grid + Solar


### Handling Duplicate Records

The duplicated Customer IDs were investigated by comparing the associated records. Each duplicated pair contains identical information across all variables, indicating that the records are exact duplicates rather than legitimate repeated observations. Therefore, the duplicate records will be removed to prevent the same customer information from being counted more than once during analysis.

In [31]:
df.duplicated().sum()

np.int64(12)

### Removing Exact Duplicate Records

The investigation confirmed that the 12 repeated Customer ID records are exact duplicate rows. These duplicate records will be removed because retaining them would cause the same customer information to be counted more than once in subsequent analyses.

In [32]:
df = df.drop_duplicates()

In [33]:
df.shape

(3000, 6)

### Part B - Step 18: Post-Cleaning Missing Value Check

After addressing the major data-quality issues, the dataset will be checked again for remaining missing values. This will confirm the effect of the cleaning steps and identify any remaining fields that require appropriate treatment before analysis.

In [34]:
df.isnull().sum()

Customer ID              0
City                     0
Customer Type           18
Monthly Consumption    190
Account Status          15
Energy Source           20
dtype: int64

In [35]:
df[df["Customer Type"].isnull() |
   df["Account Status"].isnull() |
   df["Energy Source"].isnull()].head(20)

,Customer ID,City,Customer Type,Monthly Consumption,Account Status,Energy Source
35,EN00036,Kano,Small Business,1072.6,Pending,NaN
69,EN00070,Abuja,Public Institution,1602.9,Active,NaN
76,EN00077,Kano,Residential,125.6,Active,NaN
134,EN00135,Abuja,NaN,1507.8,Active,Grid + Generator
278,EN00279,Lagos,Residential,520.7,Active,NaN
280,EN00281,Enugu,Public Institution,1008.9,Active,NaN
380,EN00381,Benin City,Small Business,998.8,Inactive,NaN
416,EN00417,Abuja,Residential,475.2,NaN,Grid + Solar
495,EN00496,Ibadan,NaN,1271.4,Active,Solar
537,EN00538,Lagos,Small Business,1166.3,NaN,Grid Electricity


### Handling Remaining Categorical Missing Values

The remaining missing values in `Customer Type`, `Account Status`, and `Energy Source` occur in otherwise usable customer records. Since the available variables do not provide sufficient information to reliably determine the missing categories, the missing values will be labelled **`Unknown`** rather than deleting the affected records or making unsupported assumptions.

In [37]:
df = df.copy()

In [38]:
df["Customer Type"] = df["Customer Type"].fillna("Unknown")
df["Account Status"] = df["Account Status"].fillna("Unknown")
df["Energy Source"] = df["Energy Source"].fillna("Unknown")

In [39]:
df.isnull().sum()

Customer ID              0
City                     0
Customer Type            0
Monthly Consumption    190
Account Status           0
Energy Source            0
dtype: int64

In [40]:
df.shape

(3000, 6)

In [41]:
df.dtypes

Customer ID             object
City                    object
Customer Type           object
Monthly Consumption    float64
Account Status          object
Energy Source           object
dtype: object

### Part C - Creating Consumption Category

A new variable, `Consumption_Category`, will be created using the business rule specified for the analysis. Customers with monthly consumption of **500 or more** will be classified as **High Consumption**, while customers with consumption below **500** will be classified as **Normal Consumption**. Records with missing consumption values will remain missing because their consumption category cannot be determined.

In [43]:
import numpy as np

In [45]:
df["Consumption_Category"] = "Normal Consumption"

df.loc[
    df["Monthly Consumption"] >= 500,
    "Consumption_Category"
] = "High Consumption"

df.loc[
    df["Monthly Consumption"].isna(),
    "Consumption_Category"
] = np.nan

In [46]:
df["Consumption_Category"].value_counts(dropna=False)

Consumption_Category
Normal Consumption    1526
High Consumption      1284
NaN                    190
Name: count, dtype: int64

### Part D - Step 1: Number of Customers in Each City

The number of customers in each city will be determined using a frequency table. This will show how the 3,000 customer records are distributed geographically across the cities represented in the dataset.

In [48]:
df["City"].value_counts()

City
Lagos            641
Abuja            455
Port Harcourt    370
Ibadan           326
Kano             307
Enugu            231
Benin City       188
Kaduna           165
Aba              145
Jos              130
Unknown           42
Name: count, dtype: int64

### Part D - Step 2: Average Monthly Consumption

The overall average monthly electricity consumption will be calculated using the valid numerical consumption records. Missing consumption values will be automatically excluded from the calculation.

In [49]:
df["Monthly Consumption"].mean()

np.float64(1079.9316014234876)

### Part D - Step 3: Number of High-Consumption Customers

The number of customers classified as **High Consumption** will be determined using the business rule established in Part C, where monthly consumption of **500 or more** is classified as high consumption. Records with missing consumption values are excluded from this count.

In [50]:
(df["Consumption_Category"] == "High Consumption").sum()

np.int64(1284)

### Part D - Step 4: Average Consumption by Customer Type

Average monthly electricity consumption will be compared across customer types to identify which customer groups have the highest and lowest average consumption. Missing consumption values will be excluded automatically from the mean calculation.

In [51]:
df.groupby("Customer Type")["Monthly Consumption"].mean()

Customer Type
Commercial            1788.779657
Industrial            6464.325773
Public Institution    1060.188722
Residential            306.492541
Small Business         736.887049
Unknown                964.652941
Name: Monthly Consumption, dtype: float64

### Part D - Step 5: Number of Customers by Energy Source

The frequency of customers using each energy source will be determined to understand the distribution of energy sources across the dataset. The `Unknown` category will be retained because it represents records where the energy source was unavailable.

In [52]:
df["Energy Source"].value_counts()

Energy Source
Grid Electricity    1653
Grid + Solar         463
Solar                334
Generator            325
Grid + Generator     205
Unknown               20
Name: count, dtype: int64

### Challenge - Customer Type Consumption Summary

A summary table will be created to compare the average monthly consumption and number of customers across customer types. This will help identify customer groups with notably different consumption patterns that may warrant further investigation.

In [53]:
customer_type_summary = df.groupby("Customer Type").agg(
    Average_Consumption=("Monthly Consumption", "mean"),
    Number_of_Customers=("Customer ID", "count")
)

customer_type_summary

,Average_Consumption,Number_of_Customers
Customer Type,,
Commercial,1788.779657,434
Industrial,6464.325773,212
Public Institution,1060.188722,143
Residential,306.492541,1542
Small Business,736.887049,651
Unknown,964.652941,18


### Recommendation

The analysis shows a substantial difference in average electricity consumption across customer types, with Industrial customers recording the highest average consumption at approximately 6,464.33 units, compared with 306.49 units among Residential customers. The energy company should therefore investigate the factors contributing to the substantially higher consumption among Industrial customers, including their operational requirements, energy-use patterns, and energy sources. Further analysis could also examine whether the high consumption observed among industrial customers is concentrated in particular cities or energy-source categories. This would provide a better basis for understanding the underlying consumption pattern before any business decision is made.